# laya RLCD training on Kaggle (parameterized)

Runtime deps with zero GitHub DNS dependency:
- laya 0.4.6 from PyPI (the exact version validated locally)
- splits from the jsdelivr GitHub mirror (Kaggle's DNS
  cannot resolve github.com; the CDN mirror is reachable)
- multilingual base = a9597e0 reconstructed: weights +
  tokenizer.json are identical LFS blobs on the current
  repo (the history rewrite changed only config files),
  and the 3 a9597e0 config files (gone from HF after
  the rewrite) are mirrored in our repo and overlaid

Headless via the push->poll->pull pattern (see docs/training-plan.md 6.1):

```bash
kaggle kernels push -p cloud/kaggle/   # upload triggers a GPU run
kaggle kernels status hathawayandrx105/train    # poll until complete
kaggle kernels output hathawayandrx105/train -p out/
```

The checkpoint is pushed to HF Hub (set the HF_TOKEN secret
in the notebook's Secrets panel); the report lands in
/kaggle/working (pulled by `kernels output`).

In [ ]:
EPOCHS = 4
UNFREEZE = False
HF_REPO = "hathawayandrx105/laya-canon-head"

In [ ]:
import pathlib
import subprocess
import urllib.request

subprocess.run(["pip", "install", "laya==0.4.6"], check=True)

# splits via the jsdelivr GitHub mirror
base = ("https://cdn.jsdelivr.net/gh/"
        "hathawayandrx105/laya-canon-judgments@main/"
        "data/splits/")
out = pathlib.Path("/kaggle/working/splits")
out.mkdir(exist_ok=True)
for name in ("train.jsonl", "validation.jsonl", "test.jsonl",
             "manifest.json"):
    urllib.request.urlretrieve(base + name, out / name)
    print("fetched", name)

In [ ]:
import os
import urllib.request

from huggingface_hub import snapshot_download

from laya.train import TrainConfig, finetune
from laya.agent import _fix_tokenizer_config

# Base = a9597e0 multilingual, reconstructed:
# - weights + tokenizer.json: identical LFS blobs on
#   the current repo (the history rewrite changed only
#   config files), so the snapshot download gets them
# - the 3 a9597e0 config files (gone from HF after
#   the rewrite) are mirrored in our repo and overlaid
snap = snapshot_download(
    "convaiinnovations/laya",
    local_dir="/kaggle/working/laya-snap",
    allow_patterns=["multilingual/**"],
)
model_dir = snap + "/multilingual"
cfg = ("https://cdn.jsdelivr.net/gh/"
       "hathawayandrx105/laya-canon-judgments@main/"
       "data/base-config/")
for rel in ("rl_agent_config.json", "encoder/config.json",
            "tokenizer/tokenizer_config.json"):
    dest = os.path.join(model_dir, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(cfg + rel, dest)
    print("overlaid", rel)
_fix_tokenizer_config(model_dir)
config = TrainConfig(
    epochs=EPOCHS, micro_batch=4, grad_accum=2,
    freeze_encoder=not UNFREEZE, seed=42, calib_seed=42,
    max_len=1024, head_max_len=256,
)
summary = finetune(
    data="/kaggle/working/splits/train.jsonl",
    model_dir=model_dir,
    output_dir="/kaggle/working/out",
    config=config,
    device="cuda",
)
print(summary)

In [ ]:
import json
import subprocess

# Acceptance on the held-out test split (6 state groups):
# base (reconstructed a9597e0) vs fine-tuned head.
for name, eval_dir in (
    ("base", model_dir),
    ("finetuned", "/kaggle/working/out/checkpoint_latest"),
):
    ev = subprocess.run(
        ["laya-evals", "run", "/kaggle/working/splits/test.jsonl",
         "--model", eval_dir, "--device", "cuda",
         "--json", f"/kaggle/working/eval-test-{name}.json"],
        capture_output=True, text=True)
    print(f"--- test eval: {name} (rc={ev.returncode})")
    print(ev.stdout[-2500:])
    if ev.returncode != 0:
        print(ev.stderr[-1500:])

In [ ]:
import json
import os

from huggingface_hub import HfApi

report = {"summary": str(summary), "epochs": EPOCHS,
          "unfreeze": UNFREEZE}
for name in ("base", "finetuned"):
    try:
        with open(f"/kaggle/working/eval-test-{name}.json") as f:
            report[f"test_{name}"] = json.load(f)
    except FileNotFoundError:
        report[f"test_{name}"] = None
with open("/kaggle/working/report.json", "w") as f:
    json.dump(report, f, indent=2)

token = os.environ.get("HF_TOKEN")
if token:
    api = HfApi(token=token)
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, folder_path="/kaggle/working/out",
                      path_in_repo="")
    print("checkpoint pushed to", HF_REPO)
else:
    print("HF_TOKEN not set; checkpoint stays in /kaggle/working/out")